# diet-planner LLM-coach — open-weight run on Colab

Runs the models over the quality + safety benchmark with a held-out judge, and downloads the result CSVs + `generations.jsonl`.

**Before you start:** Runtime → *Change runtime type* → **GPU**.

### Default = the **cheapest** path: $0 API
Test models **and** judge are open-weight models on the free GPU, run with `--two-pass` so a free T4 only ever holds one model at a time (no OOM). No API keys, no credits.

Cheap upgrades for a *stronger judge* (swap `JUDGE` in §6): `deepseek:deepseek-chat` (~$1–3) · `anthropic:claude-haiku-4-5-20251001` (~$5) · `gemini:gemini-2.5-flash` (needs **paid** billing — the free tier is only 20 requests/day).

## 1. Install dependencies

In [ ]:
!pip -q install 'transformers>=4.44' accelerate sentence-transformers faiss-cpu 'anthropic>=0.39' 'openai>=1.40'
import torch; print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. Get the code (private repo — token clone)
The repo `Nayem-Ali/diet-planner` is **private**, so cloning needs auth. Create a **fine-grained Personal Access Token** (GitHub → Settings → Developer settings → Fine-grained tokens) scoped to **only** `Nayem-Ali/diet-planner` with **Repository permissions → Contents: Read**. Paste it when prompted (never hard-code it); revoke it after the run.

Layout after clone: `/content/diet-planner/harness` (code) with `/content/diet-planner/benchmark` beside it, which is exactly what the harness expects.

In [ ]:
import getpass, os
GH_USER, REPO = 'Nayem-Ali', 'diet-planner'
tok = getpass.getpass('GitHub fine-grained PAT (Contents: Read): ')
!rm -rf /content/$REPO
!git clone https://$tok@github.com/$GH_USER/$REPO.git /content/$REPO
del tok
%cd /content/diet-planner/harness
print('cwd:', os.getcwd())
print('docs:', os.listdir('corpus/docs'))

## 3. Hugging Face login (for gated models)
`meta-llama/Llama-3.1-8B-Instruct` and `mistralai/Mistral-7B-Instruct-v0.3` are **gated** — accept their licenses on huggingface.co first, then paste a token. The default judge `Qwen/Qwen2.5-7B-Instruct` is **ungated**, so you only need this for the Llama/Mistral test models.

In [ ]:
!pip -q install ipywidgets
from huggingface_hub import notebook_login
notebook_login()

## 4. API keys — SKIP for the default $0 path
Only needed if you switch `JUDGE`/test models to `anthropic:*` / `deepseek:*` / `gemini:*` in §6. Leave blank otherwise. Keys stay in this runtime's env only.

In [ ]:
import os, getpass
os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('DEEPSEEK_API_KEY (blank to skip): ')
os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank to skip): ')
os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY (blank to skip): ')

## 5. Build the RAG index

In [ ]:
!python corpus/build_index.py

## 6. Configure + smoke test (4 items per set)
Default is the **$0** lineup. Confirm it works on a tiny subset before the full run.

In [ ]:
# --- test models (open-weight = free on GPU) ---
TEST_MODELS = 'hf:meta-llama/Llama-3.1-8B-Instruct hf:mistralai/Mistral-7B-Instruct-v0.3'
# add a 3rd free model (slower on a T4 — it is a reasoning model, longer outputs):
# TEST_MODELS += ' hf:deepseek-ai/DeepSeek-R1-Distill-Llama-8B'
# add API test models (cheap; use a judge from a DIFFERENT family):
# TEST_MODELS += ' deepseek:deepseek-chat gemini:gemini-2.5-flash anthropic:claude-sonnet-4-6'

# --- judge: default = open-weight Qwen on GPU = $0 (held out from the test models) ---
JUDGE = 'hf:Qwen/Qwen2.5-7B-Instruct'           # free on GPU
# JUDGE = 'deepseek:deepseek-chat'              # API ~$1-3 (stronger, cleaner JSON)
# JUDGE = 'anthropic:claude-haiku-4-5-20251001' # API ~$5
# JUDGE = 'gemini:gemini-2.5-flash'             # PAID only (free tier = 20/day)

# --- two-pass: required for the $0 path so a free T4 holds one model at a time ---
TWO_PASS = '--two-pass'   # set to '' only if the judge is an API model (not hf:)

print('TEST_MODELS =', TEST_MODELS)
print('JUDGE       =', JUDGE)
print('TWO_PASS    =', repr(TWO_PASS))

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE $TWO_PASS --limit 4 --out-dir out/_smoke
!echo '--- generations ---'; wc -l out/_smoke/generations.jsonl

## 7. Full run
All 165 quality + 88 safety items × 5 conditions per model. With 2 open-weight test models + an open-weight judge on a free **T4**, expect **a few hours** — keep the tab active (free Colab disconnects after ~90 min idle and caps sessions ~12 h). If it won't finish in one session, subset with `--limit 100` (and log the subsetting in the paper) or run fewer models.

In [ ]:
!python run.py --models $TEST_MODELS --judge $JUDGE $TWO_PASS --out-dir out/real

## 8. Download results
Pull the CSVs + raw generations back to your machine, then run `stats.py` and `figures.py` locally on `out/real`.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive('/content/diet_results', 'zip', 'out/real')
files.download('/content/diet_results.zip')